# Task 3 — Notebook 01: SMAP Data Loading

**Purpose (NAFSI Task 3):** **Soil moisture anomalies** for a chosen event window vs. a multi-year baseline, with CDL for agricultural context. This notebook loads **all weekly SMAP interim years** as one array and loads **CDL** for the mask year from the interim CDL stack.

**Inputs (interim):**  
- `data/interim/smap/smap_weekly_{year}.nc` — combined along `calendar_year`  
- `data/interim/cdl/cdl_stack_*.nc` — slice `year` = `cdl_mask.year` from config

**Outputs:** in-memory `smap_all`, optional slices `smap_baseline`, `smap_event`, and `cdl_mask` for later climatology / anomaly notebooks.

In [ ]:
# NOTE: This notebook was scaffolded with AI assistance.

import sys
from pathlib import Path

import pandas as pd
import xarray as xr
import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root. cd to the repo and retry.")
sys.path.insert(0, str(REPO_ROOT))

from src.io.interim_loaders import load_cdl_stack_from_interim, load_smap_weekly_all_years

with open(REPO_ROOT / "configs" / "task3_soil_moisture.yaml") as f:
    cfg = yaml.safe_load(f)

b0, b1 = int(cfg["baseline"]["period"][0]), int(cfg["baseline"]["period"][1])
mask_year = int(cfg["cdl_mask"]["year"])
print("Baseline years:", b0, "–", b1, "| CDL mask year:", mask_year)

In [ ]:
smap_all = load_smap_weekly_all_years(REPO_ROOT)
smap_baseline = smap_all.sel(calendar_year=slice(b0, b1))

ev = cfg["event_window"]
t0, t1 = pd.Timestamp(ev["start_date"]), pd.Timestamp(ev["end_date"])

def _slice_event(da):
    t = da["time"]
    if not hasattr(t.dtype, "tz") or t.dtype.tz is None:
        t_cmp = t
        t0c, t1c = t0, t1
    else:
        t_cmp = t
        t0c, t1c = t0.tz_localize(None), t1.tz_localize(None)
    return da.sel(time=(t_cmp >= t0c) & (t_cmp <= t1c))


smap_event_parts = []
for y in smap_all["calendar_year"].values:
    sub = _slice_event(smap_all.sel(calendar_year=int(y)))
    if sub.sizes.get("time", 0) > 0:
        smap_event_parts.append(sub)
smap_event = xr.concat(smap_event_parts, dim="time").sortby("time") if smap_event_parts else None

cdl_full = load_cdl_stack_from_interim(REPO_ROOT)
if mask_year not in set(int(y) for y in cdl_full["year"].values):
    raise ValueError(f"CDL mask year {mask_year} not in stack: {cdl_full['year'].values}")
cdl_mask = cdl_full.sel(year=mask_year)

print("smap_all", dict(smap_all.sizes))
print("smap_baseline", dict(smap_baseline.sizes))
print("cdl_mask", dict(cdl_mask.sizes))
if smap_event is not None:
    print("smap_event (concat over years)", dict(smap_event.sizes))


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Mean surface SM for one baseline year (first available in slice)
if smap_baseline.sizes.get("calendar_year", 0) == 0:
    print("No SMAP weeks in baseline slice — check interim files and baseline.period.")
else:
    by0 = int(smap_baseline["calendar_year"].values[0])
    m = smap_baseline.sel(calendar_year=by0).mean(dim="time")
    fig, ax = plt.subplots(figsize=(6, 5))
    m.plot(ax=ax, robust=True)
    ax.set_title(f"SMAP weekly mean ({by0}, weeks in file)")
    plt.tight_layout()
